In [ ]:
# 字体配置：TOOLS.md 唯一标准（所有 notebook 必须用这个）
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"

font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()

import matplotlib.pyplot as plt
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("使用字体:", font_name)

In [ ]:
import os, re, glob, json, subprocess
from collections import Counter, OrderedDict

LNK = "/root/lnkcre"
MIG_MYSQL = os.path.join(LNK, "backend/internal/platform/database/migrations")
MIG_PG = os.path.join(LNK, "backend/internal/platform/database/migrations-pg")
CANON = os.path.join(LNK, "backend/internal/platform/database/testdata/canonical_tables.txt")

tables = [l.strip() for l in open(CANON) if l.strip()]
print("canonical tables:", len(tables))
print("样例:", tables[:5], "...")

In [ ]:
# 解析双迁移体系：容忍正则，PG 双引号标识符去引号 + 小写
CT = re.compile(r'create\s+table\s+(?:if\s+not\s+exists\s+)?["`]?([a-zA-Z_0-9]+)', re.I)

creations = []      # (table, system, seq, path) —— 全量创建记录，供增长分析
first_seen = {}     # table -> (system, base, fname) —— MySQL 优先的首次创建

def parse_migs(d, system):
    out = {}
    for f in sorted(glob.glob(os.path.join(d, "*.up.sql"))):
        fname = os.path.basename(f)
        seq = int(fname.split("_")[0])
        base = re.sub(r"^\d+_", "", fname).replace(".up.sql", "")
        txt = open(f, encoding="utf-8", errors="replace").read()
        for m in CT.finditer(txt):
            t = m.group(1).lower()
            creations.append((t, system, seq, f))
            if t not in out:
                out[t] = (system, base, fname)
    return out

mm = parse_migs(MIG_MYSQL, "mysql")
mp = parse_migs(MIG_PG, "pg")

first_seen = dict(mm)
for t, v in mp.items():
    first_seen.setdefault(t, v)

missing = [t for t in tables if t not in first_seen]
print(f"MySQL 迁移解析表数: {len(mm)}")
print(f"PG   迁移解析表数: {len(mp)}（200 个 up.sql 文件）")
print(f"并集: {len(first_seen)}")
print(f"canonical 缺失（并集中没有）: {len(missing)} {missing}")
assert len(tables) == 472 and len(missing) == 0, "canonical 规模或覆盖异常，停止后续分析"
print("\n结构发现：双迁移体系并集 = canonical 全覆盖（0 缺失），PG 侧是全量表基线，MySQL 侧是活跃子集。")

In [ ]:
P = "Platform/Shared"

# ---- 基线规则（46 条，按序匹配，先 specific 后 generic）----
RULES_CORE = [
    (r"^office_energy|^meter_monthly", "12 Engineering & Facility", ""),
    (r"engineering_condition", "01 Asset Foundation", ""),
    (r"chart_of_account", "09 Accounting Bridge", ""),
    (r"^office_?(lead|profile|hold)", "03 Leasing Pipeline", ""),
    (r"^park", "16 Parking Management", ""),
    (r"parking", "16 Parking Management", ""),
    (r"^overtime", "16 Parking Management", "overtime 停车超时费归16有灰度"),
    (r"^intent_deposit", "07 Collection & Settlement", ""),
    (r"^quotation", "03 Leasing Pipeline", ""),
    (r"^lease_amendment|^amendment", "04 Contract Lifecycle", ""),
    (r"^lease_contract|^lease_lifecycle|^lease_pilot|^lease_allocation|^lease_contracts", "05 Lease/Occupancy", ""),
    (r"^lease_discount", "05 Lease/Occupancy", ""),
    (r"^lease_template|^contract_template", "04 Contract Lifecycle", ""),
    (r"customer_follow_up", "03 Leasing Pipeline", ""),
    (r"^tenant_(bills|breach)", "06 Billing & AR", "tenant_breach 归06还是04有灰度"),
    (r"^tenant_(daily_sales|sales)", "17 BI & Analytics", "销售采集归17还是02有灰度"),
    (r"^tenant_(messages|service_requests?)", "13 Work Order Service", ""),
    (r"^tenant_(activities|home)", "10 Operations Management", ""),
    (r"^tenant_", "02 Merchant", ""),
    (r"^sales_|_sales$|^daily_sales|^seller", "17 BI & Analytics", "sales 事实表归17有灰度"),
    (r"^deposit", "07 Collection & Settlement", ""),
    (r"^invoice|^tax", "08 Tax Invoice", ""),
    (r"^voucher|^accounting|^chart|^commission|^accrual", "09 Accounting Bridge", ""),
    (r"^finance_(closed|masterdata|stub)", "09 Accounting Bridge", ""),
    (r"^settlement|^receipt|^collection|^payment|^dunning|^writeoff|^reconciliation|^advance_transfer|^take_high", "07 Collection & Settlement", ""),
    (r"^billing|^charge|^fee|^ar_|^receivable|^expected_income|^credit_note|^interest|^budget|^pricing|^con_00|^late_fee|^free_rent|^category_rate", "06 Billing & AR", ""),
    (r"^alert", "17 BI & Analytics", "alert 归17（预警消费在BI侧）有灰度"),
    (r"^analytics|^dashboard|^report|^dataset|^chart|^bigscreen|^filter_view|^wire_print", "17 BI & Analytics", ""),
    (r"^equipment|^meter|^iot|^energy|^hazard|^safety|^material|^maintenance|^smart_device|^tariff|^renovation|^inspection", "12 Engineering & Facility", ""),
    (r"^cleaning|^patrol|^property_task|^certificate|^confidentiality", "11 Property Management", ""),
    (r"^service_request|^tenantrequest|^servicerequest", "13 Work Order Service", ""),
    (r"^promotion|^footfall|^brand_placement|^marketing|^campaign|^multibiz|^resource_booking", "14 Marketing & Campaign", ""),
    (r"^member|^customer_referral|^coupon|^points", "15 Customer/Member", ""),
    (r"^customer|^brand", "02 Merchant", ""),
    (r"^merchant|^seller", "02 Merchant", ""),
    (r"^broker", "03 Leasing Pipeline", ""),
    (r"^leasing|^opportunity|^prospect|^funnel|^lead|^intent|^channel_viewing", "03 Leasing Pipeline", ""),
    (r"^condition_approval", "04 Contract Lifecycle", ""),
    (r"contract|^amendment|^renewal|^party_rule|^draft|^company_resolution|^credit_note_l", "04 Contract Lifecycle", ""),
    (r"^occupancy", "05 Lease/Occupancy", ""),
    (r"^shop|^building|^floor|^structure|^warehouse|^unified_resource|^resource_floor|^units?_|^project|^multi_business|^unit_|^split_merge", "01 Asset Foundation", ""),
    (r"^operations|^operational|^shift|^dutylog|^overtime_billing", "10 Operations Management", ""),
    (r"^auth|^org|^menu|^user|^role|^permission|^refresh_token|^functions|^audit_log|^access_audit|^oplog|^operation_log|^notification|^config|^dict|^custom_field|^custom_form|^workflow|^coding|^doc_no|^auto_increment|^print|^docoutput|^document_delivery|^mobile_portal|^i18n|^listpref|^datascope|^system_mgmt|^app_|^modules|^integration|^e2e|^platform|^http|^logging|^pagination|^sqlutil|^testutil|^money|^draft$|^ai_draft|^operation|^permission_functions|^position_pricing", P, ""),
    (r"^bank", "09 Accounting Bridge", "银企直联归09有灰度"),
]

# ---- 9/6 补录：孤儿收敛规则（每条有包 grep 或迁移名证据）----
RULES_FIX = [
    (r"^fill_cell|^fill_session", "17 BI & Analytics", ""),                                  # 包证据: reportdesigner/fill
    (r"^fixed_asset", "12 Engineering & Facility", "固定资产台账：实物口径归12，财务口径可改09，有灰度"),  # 包证据: fixedasset
    (r"^wf_", P, "工作流引擎表：业务审批语义在04，引擎归Platform，有灰度"),                    # 包证据: workflow/outbox|publication
    (r"^approval_authority", "04 Contract Lifecycle", "审批权限矩阵：与condition-approval同族，有灰度"),  # 包证据: approvalmatrix
    (r"^lease_party", "04 Contract Lifecycle", ""),                                           # 包证据: partysplit
    (r"^temporary_charge", "06 Billing & AR", "settlement词根亦指07，有灰度"),                  # 包证据: temporarycharge
    (r"^industry_dict", P, ""),                                                               # 包证据: industrydict（字典）
    (r"^pos_payment", "07 Collection & Settlement", "无代码引用，疑似预留表",),
    (r"^spatial_project", "01 Asset Foundation", "backfill/quarantine为数据治理日志，有灰度"),
]
RULES = RULES_CORE + RULES_FIX

def classify(name, rules):
    n = name.lower()
    for pat, ctx, gray in rules:
        if re.search(pat, n):
            return ctx, pat, gray
    return None, "", ""

# L1/L3（core 规则）先跑一遍，收集待 L2 的表
rec_core = {}
pending = []
for t in tables:
    src = first_seen.get(t)
    base = src[1] if src else t
    ctx, pat, gray = classify(base, RULES_CORE)
    prov = "L1:migration"
    if ctx is None and src:
        ctx, pat, gray = classify(t, RULES_CORE)
        prov = "L3:tablename"
    if ctx is None:
        pending.append(t)
    else:
        rec_core[t] = (ctx, prov, pat, gray)
print("L1/L3（core 规则）后待定表:", len(pending))

In [ ]:
import time
t0 = time.time()

# L2 流式包扫描：只为待定表建联合正则，单遍扫 Go 源码（避免全量索引）
pend_pat = {t: re.compile(r"(?<![a-zA-Z_0-9])" + re.escape(t) + r"(?![a-zA-Z_0-9])") for t in pending}
COMBINED = re.compile("|".join(re.escape(t) for t in pending)) if pending else None
hits = {t: set() for t in pending}
n_files = 0
for root, dirs, files in os.walk(os.path.join(LNK, "backend/internal")):
    dirs[:] = [d for d in dirs if d not in ("testdata", "node_modules")]
    for f in files:
        if not f.endswith(".go") or f.endswith("_test.go"):
            continue
        n_files += 1
        try:
            txt = open(os.path.join(root, f), encoding="utf-8", errors="replace").read()
        except OSError:
            continue
        if COMBINED and COMBINED.search(txt):
            for t in pending:
                if pend_pat[t].search(txt):
                    hits[t].add(os.path.basename(root))
print(f"扫描 {n_files} 个 Go 文件，耗时 {time.time()-t0:.1f}s，命中待定表 {sum(1 for t in pending if hits[t])}/{len(pending)}")

# ---- 基线（仅 core 规则）孤儿 ----
rec = dict(rec_core)
orphans_base = []
for t in pending:
    done = False
    for p in sorted(hits[t], key=lambda x: -len(x)):
        c2, p2, g2 = classify(p, RULES_CORE)
        if c2:
            rec[t] = (c2, "L2:package", "pkg:" + p, g2)
            done = True
            break
    if not done:
        orphans_base.append(t)
print(f"\n基线孤儿（core 规则 + L2 兜底）: {len(orphans_base)} 张")
for t in orphans_base:
    print("  ", t, "| 包证据:", sorted(hits.get(t, [])) or "无引用")

In [ ]:
# ---- 应用补录规则后的终局归类 ----
recf = {}
orphans_final = []
for t in tables:
    src = first_seen.get(t)
    base = src[1] if src else t
    ctx, pat, gray = classify(base, RULES)
    prov = "L1:migration"
    if ctx is None and src:
        ctx, pat, gray = classify(t, RULES)
        prov = "L3:tablename"
    if ctx is None:
        for p in sorted(hits.get(t, ()), key=lambda x: -len(x)):
            c2, p2, g2 = classify(p, RULES)
            if c2:
                ctx, pat, gray, prov = c2, "pkg:" + p, g2, "L2:package"
                break
    if ctx is None:
        orphans_final.append(t)
        ctx = "ORPHAN"
    recf[t] = (ctx, prov, pat, gray)

print(f"孤儿收敛：基线 {len(orphans_base)} → 终局 {len(orphans_final)}")
assert not orphans_final, f"仍有孤儿: {orphans_final}"

cnt = Counter(v[0] for v in recf.values())
prov_cnt = Counter(v[1] for v in recf.values())
total = len(tables)
print(f"\n各 Context 表数（共 {total} 张，18 个桶 = 17 Context + Platform/Shared）:")
for k, v in sorted(cnt.items(), key=lambda x: -x[1]):
    print(f"  {v:4d} ({v/total*100:4.1f}%)  {k}")
print("\nprovenance:", dict(prov_cnt))
contexts = [c for c in cnt if c != P]
print(f"\n覆盖率: 17/17 Context 均有表；Platform/Shared 另有 {cnt[P]} 张 ({cnt[P]/total*100:.1f}%)")

In [ ]:
# 图1：Context 分布（Platform/Shared 单独配色以示“非业务 Context”）
order = sorted(cnt.items(), key=lambda x: x[1])
labels = [k for k, _ in order]
vals = [v for _, v in order]
colors = ["#888888" if k == P else "#2b6cb0" for k in labels]

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(labels, vals, color=colors)
for b, v in zip(bars, vals):
    ax.text(v + 0.5, b.get_y() + b.get_height()/2, str(v), va="center", fontsize=9)
ax.set_xlabel("表数量")
ax.set_title(f"472 张 canonical 表 × MI Domain Model 17 Context（灰 = Platform/Shared 非业务桶）")
ax.set_xlim(0, max(vals) * 1.12)
plt.tight_layout()
plt.savefig("/root/learning-notebooks/第14周/w14d5_context分布.png", dpi=150)
plt.show()

In [ ]:
out = subprocess.run(
    ["git", "-C", LNK, "log", "--diff-filter=A", "--name-only",
     "--pretty=format:%ad", "--date=format:%Y-%m", "--", MIG_MYSQL, MIG_PG],
    capture_output=True, text=True).stdout

file_month = {}
cur = None
for line in out.splitlines():
    line = line.strip()
    if not line:
        cur = None
        continue
    if re.fullmatch(r"\d{4}-\d{2}", line):
        cur = line
        continue
    if cur:
        file_month[line] = cur

table_month = {}
undated = []
for t, system, seq, path in creations:
    m = file_month.get(os.path.relpath(path, LNK))  # git pathspec 输出是仓库相对路径
    if m is None:
        continue
    if t not in table_month or m < table_month[t]:
        table_month[t] = m
undated = [t for t in tables if t not in table_month]

months = sorted(set(table_month.values()))
assert len(months) >= 1, "git 月份解析为空，检查 file_month/path 匹配"
ctx_order = [k for k, _ in sorted(cnt.items(), key=lambda x: -x[1])]
mat = {m: {c: 0 for c in ctx_order} for m in months}
for t, m in table_month.items():
    mat[m][recf[t][0]] += 1

print("月份序列:", months, "| 无日期表:", len(undated), undated[:8])
tot = {m: sum(mat[m].values()) for m in months}
cum, acc = [], 0
for m in months:
    acc += tot[m]
    cum.append(acc)
print("月度新增表:", tot)
print("累计:", dict(zip(months, cum)))

In [ ]:
# 图2：月度 × Context 增长热力图
import numpy as np
data = np.array([[mat[m][c] for c in ctx_order] for m in months])

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(data, aspect="auto", cmap="YlGnBu")
ax.set_xticks(range(len(ctx_order)))
ax.set_xticklabels([c.split(" ", 1)[0] for c in ctx_order], rotation=0, fontsize=9)
ax.set_yticks(range(len(months)))
ax.set_yticklabels([f"{m}\n(+{tot[m]})" for m in months], fontsize=9)
for i in range(len(months)):
    for j in range(len(ctx_order)):
        v = data[i, j]
        if v:
            ax.text(j, i, str(v), ha="center", va="center", fontsize=8,
                    color="white" if v > data.max()*0.55 else "black")
ax.set_title("各月新增表 × Context（Context 按总量降序；右侧色条 = 当月该 Context 新表数）")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig("/root/learning-notebooks/第14周/w14d5_月度增长热力图.png", dpi=150)
plt.show()

In [ ]:
# 机器可读报告：w14d5-context-coverage-report.yaml
import yaml, datetime

gray_zones = []
for pat, ctx, gray in RULES:
    if gray:
        gray_zones.append({"pattern": pat, "context": ctx, "note": gray})

by_context = []
for c in ctx_order:
    ts = sorted(t for t in tables if recf[t][0] == c)
    by_context.append({
        "context": c,
        "tables": len(ts),
        "share_pct": round(len(ts)/total*100, 1),
        "provenance": dict(Counter(recf[t][1] for t in ts)),
        "table_list": ts,
    })

report = {
    "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "experiment": "W14-D5 DomainModel 覆盖率检查（实验2）",
    "inputs": {
        "canonical_tables": CANON,
        "canonical_count": total,
        "migrations_mysql": MIG_MYSQL,
        "migrations_pg": MIG_PG,
        "lineage_note": "计划记载 336 已过时；实际 307→336→472（+136/4 个月）",
    },
    "method": {
        "layers": {
            "L1": "迁移文件基名规则（55 条中的 46 条 core）",
            "L3": "表名规则兜底",
            "L2": "包名 grep（代码证据，仅扫待定表，单遍流式）",
        },
        "rules_total": len(RULES),
        "rules_core": len(RULES_CORE),
        "rules_fix_0906": len(RULES_FIX),
        "orphan_convergence": {"baseline": len(orphans_base), "final": len(orphans_final)},
    },
    "coverage": {
        "contexts_with_tables": "17/17",
        "platform_bucket_tables": cnt[P],
        "platform_share_pct": round(cnt[P]/total*100, 1),
        "orphan_tables": orphans_final,
        "provenance": dict(prov_cnt),
    },
    "dual_migration_finding": {
        "mysql_tables": len(mm),
        "pg_files": len(glob.glob(os.path.join(MIG_PG, '*.up.sql'))),
        "pg_tables": len(mp),
        "union": len(first_seen),
        "canonical_missing": len(missing),
        "note": "并集=canonical 全覆盖；PG 为全量基线、MySQL 为活跃子集；双体系并存是 schema 演进分叉风险",
    },
    "by_context": by_context,
    "gray_zones": gray_zones,
    "monthly_growth": {
        m: {"new_tables": tot[m], "cumulative": c, "by_context": {c: v for c, v in mat[m].items() if v}}
        for m, c in zip(months, cum)
    },
    "references": {
        "d3_report": "/root/learning-notebooks/第14周/w14d3-ontology-health-report.yaml",
        "ontology": "/root/docs/lanlnk/config/ontology/business-ontology.yaml",
        "charts": [
            "/root/learning-notebooks/第14周/w14d5_context分布.png",
            "/root/learning-notebooks/第14周/w14d5_月度增长热力图.png",
        ],
    },
}

REPORT_PATH = "/root/learning-notebooks/第14周/w14d5-context-coverage-report.yaml"
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(report, f, allow_unicode=True, sort_keys=False, width=120)
print("已写出:", REPORT_PATH)
print("大小:", os.path.getsize(REPORT_PATH), "字节 | by_context:", len(by_context), "| gray_zones:", len(gray_zones))